# 特征空间分析

输出 `results/` 下的 `pc_association.csv`、`explained_variance.csv`、`rank_axis.csv` 和 `rank_axis/<model>.npy`。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from fmspace.analysis import association, compare_with_pc1, l2, pca, rank_axis
from fmspace.data import load_metadata, patient_split
from fmspace.extract import load_features
from fmspace.models import MODELS

DATA_DIR = Path(r"C:\M1\master paper\DataSet\DeepDRiD")
FEATURE_DIR = Path("../features")
RESULT_DIR = Path("../results")
(RESULT_DIR / "rank_axis").mkdir(parents=True, exist_ok=True)

c:\M1\envs\fmspace\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
metadata = load_metadata(DATA_DIR)
split = patient_split(metadata, seed=0)
X = {name: l2(load_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"])) for name in MODELS}
pcas = {name: pca(X[name]) for name in MODELS}

## PC × 标签关联、解释方差

PCA 用全部 1,189 张拟合（不用标签）。

In [3]:
assoc = pd.concat([association(pcas[n].transform(X[n]), metadata).assign(model=n) for n in MODELS])
variance = pd.concat(
    [pd.DataFrame({"model": n, "component": np.arange(1, 51), "ratio": pcas[n].explained_variance_ratio_}) for n in MODELS]
)
assoc.to_csv(RESULT_DIR / "pc_association.csv", index=False)
variance.to_csv(RESULT_DIR / "explained_variance.csv", index=False)

## rank axis

按 Sonthalia et al. (NeurIPS 2025) 的流程训练。`rank_rho`、`pc1_rho`、`diff` 都在 seed 0 的测试集上计算，`diff_low`、`diff_high` 为患者 bootstrap 95% 区间。

In [4]:
y = metadata["eye_dr_level"].to_numpy()
test = (split == "test").to_numpy()
rows = []
for name in MODELS:
    r = rank_axis(X[name], y, split)
    np.save(RESULT_DIR / "rank_axis" / f"{name}.npy", r["weight"])
    score = X[name] @ r["weight"]
    pc1 = pcas[name].transform(X[name])[:, 0]
    rows.append(
        {"model": name, "lr": r["lr"], "weight_decay": r["weight_decay"], "val_rho": r["val_rho"],
         **compare_with_pc1(score[test], pc1[test], y[test], metadata["patient_id"][test])}
    )
summary = pd.DataFrame(rows)
summary.to_csv(RESULT_DIR / "rank_axis.csv", index=False)
summary

,model,lr,weight_decay,val_rho,rank_rho,pc1_rho,diff,diff_low,diff_high
0,siglip,0.096839,0.000088,0.805864,0.845527,0.750277,0.095250,0.028303,0.178816
1,medsiglip,0.096839,0.000088,0.825205,0.854689,0.802084,0.052605,0.007923,0.129951
2,conceptclip,0.096839,0.000088,0.814651,0.846591,0.714632,0.131959,0.070905,0.203890
3,flair,0.096839,0.000088,0.787416,0.817523,0.799559,0.017963,-0.041104,0.085856


## 图

只读 `results/` 下的文件，不用重跑上面的计算（需要先运行第一格）。

### PC × 标签热图

In [ ]:
from fmspace.plots import heatmap

FIGURE_DIR = Path("../figures")
assoc = pd.read_csv(RESULT_DIR / "pc_association.csv")
for name in MODELS:
    fig = heatmap(assoc, name, ["eye_dr_level", "artifact", "clarity", "eye"])
    fig.savefig(FIGURE_DIR / f"heatmap_{name}.pdf", bbox_inches="tight")

### 累计解释方差

In [ ]:
from fmspace.plots import variance_curve

variance = pd.read_csv(RESULT_DIR / "explained_variance.csv")
fig = variance_curve(variance, list(MODELS), n_pcs=50, mark=20)
fig.savefig(FIGURE_DIR / "variance_cumulative.pdf", bbox_inches="tight")